In [ ]:
!pip install -q -U transformers accelerate
!pip install -q -U langchain langchain-community langchain-huggingface langchain-classic
!pip install -q pandas numpy scipy matplotlib seaborn openpyxl gradio
!pip install gptqmodel

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
 
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-AWQ"
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
 
print(f"✅ Đã load {MODEL_NAME} lên {model.device}")

In [ ]:
from langchain_huggingface import HuggingFacePipeline
 
hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,     
    do_sample=False,          # tắt sampling -> output ổn định hơn cho parsing ReAct
    repetition_penalty=1.15,  # tăng nhẹ để giảm hiện tượng lặp y chang 1 đoạn nhiều lần
    return_full_text=False,   # chỉ trả phần model sinh ra, không lặp lại prompt
)
 
llm = HuggingFacePipeline(pipeline=hf_pipe)
print("✅ LangChain LLM wrapper sẵn sàng (không qua server, gọi model trực tiếp)")

In [ ]:
import pandas as pd
import numpy as np
 
class DataStore:
    """Singleton lưu các dataset đang active trong session."""
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.datasets = {}
            cls._instance.history = []
        return cls._instance
 
    def add(self, name, df):
        self.datasets[name] = df
 
    def get(self, name):
        if name not in self.datasets:
            raise ValueError(f"Dataset '{name}' không tồn tại. Hiện có: {list(self.datasets.keys())}")
        return self.datasets[name]
 
    def log(self, action):
        self.history.append(action)
 
store = DataStore()

In [ ]:
from langchain_core.tools import tool
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats
import re
 
# QUAN TRỌNG: prompt ReAct dạy model viết Action Input dạng "key=value, key2=value2"
# (1 string). Nhưng @tool với NHIỀU tham số sẽ tạo ra StructuredTool cần 1 dict nhiều
# field. Khi LangChain nhận 1 string cho tool nhiều field, nó KHÔNG tự parse "key=value"
# -> nó nhồi nguyên string vào field ĐẦU TIÊN, các field còn lại bị thiếu -> ValidationError
# "Field required". Nên mọi tool nhiều tham số dưới đây đổi sang nhận 1 string duy nhất
# (tool_input), rồi tự parse bằng hàm dưới đây - khớp đúng format model đang generate.
def _parse_kv_string(s: str) -> dict:
    """Parse 'key1=value1, key2=value2' -> dict. Dùng regex tìm các vị trí 'key=' để
    tách value chính xác (value có thể chứa dấu phẩy hoặc dấu '=', ví dụ condition='age >= 30')."""
    s = (s or "").strip()
    if not s:
        return {}
    matches = list(re.finditer(r"(\w+)\s*=\s*", s))
    if not matches:
        return {}
    out = {}
    for i, m in enumerate(matches):
        key = m.group(1)
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(s)
        value = s[start:end].rstrip(", ").strip().strip('"').strip("'")
        out[key] = value
    return out
 
 
@tool
def create_sample_dataset(tool_input: str) -> str:
    """Tạo dataset mẫu để test. Action Input dạng: dataset_type=sales, dataset_name=ten_dataset
    dataset_type phải là 1 trong: 'sales', 'customers', 'timeseries', 'survey'.
    dataset_name là tên để lưu dataset đó, dùng cho các tool khác sau này."""
    args = _parse_kv_string(tool_input)
    dataset_type = args.get("dataset_type", "")
    dataset_name = args.get("dataset_name", "")
    if not dataset_name:
        return "Lỗi: thiếu dataset_name. Action Input cần dạng: dataset_type=sales, dataset_name=ten_dataset"
    np.random.seed(42)
    n = 200
    if dataset_type == "sales":
        df = pd.DataFrame({
            "date": pd.date_range("2024-01-01", periods=n, freq="D"),
            "region": np.random.choice(["North", "South", "East", "West"], n),
            "product": np.random.choice(["A", "B", "C"], n),
            "revenue": np.random.normal(1000, 300, n).round(2),
            "units_sold": np.random.poisson(20, n),
        })
    elif dataset_type == "customers":
        df = pd.DataFrame({
            "customer_id": range(1, n + 1),
            "age": np.random.randint(18, 70, n),
            "spend": np.random.exponential(200, n).round(2),
            "satisfaction": np.random.randint(1, 6, n),
        })
    elif dataset_type == "timeseries":
        df = pd.DataFrame({
            "date": pd.date_range("2024-01-01", periods=n, freq="D"),
            "value": np.cumsum(np.random.randn(n)) + 100,
        })
    elif dataset_type == "survey":
        df = pd.DataFrame({
            "respondent_id": range(1, n + 1),
            "score": np.random.randint(1, 11, n),
            "group": np.random.choice(["control", "treatment"], n),
        })
    else:
        return f"Lỗi: dataset_type '{dataset_type}' không hợp lệ. Dùng: sales, customers, timeseries, survey"
 
    store.add(dataset_name, df)
    store.log(f"Created sample dataset '{dataset_name}' (type={dataset_type})")
    return f"Đã tạo dataset '{dataset_name}' với {len(df)} dòng, cột: {list(df.columns)}"
 
 
@tool
def load_csv(tool_input: str) -> str:
    """Load 1 file CSV từ đường dẫn (path) vào hệ thống. Action Input dạng:
    file_path=duong/dan/file.csv, dataset_name=ten_dataset"""
    args = _parse_kv_string(tool_input)
    file_path = args.get("file_path", "")
    dataset_name = args.get("dataset_name", "")
    if not file_path or not dataset_name:
        return "Lỗi: thiếu file_path hoặc dataset_name. Action Input cần dạng: file_path=..., dataset_name=..."
    try:
        df = pd.read_csv(file_path)
        store.add(dataset_name, df)
        store.log(f"Loaded CSV '{file_path}' as '{dataset_name}'")
        return f"Đã load '{dataset_name}': {len(df)} dòng, cột: {list(df.columns)}"
    except Exception as e:
        return f"Lỗi load file: {str(e)}"
 
 
@tool
def list_available_datasets() -> str:
    """Liệt kê tất cả dataset đang có trong hệ thống."""
    if not store.datasets:
        return "Chưa có dataset nào được load."
    return "\n".join(f"- {name}: {len(df)} dòng, cột {list(df.columns)}" for name, df in store.datasets.items())
 
 
@tool
def describe_dataset(tool_input: str) -> str:
    """Thống kê mô tả đầy đủ (describe) cho 1 dataset.
    Action Input dạng: dataset_name=ten_dataset"""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", tool_input.strip())  # fallback nếu model chỉ gõ tên thẳng
    df = store.get(dataset_name)
    return df.describe(include="all").to_string()
 
 
@tool
def correlation_analysis(tool_input: str) -> str:
    """Tính ma trận tương quan giữa các cột số. Action Input dạng:
    dataset_name=ten_dataset, method=pearson (hoặc spearman, mặc định pearson)."""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", tool_input.strip())
    method = args.get("method", "pearson")
    df = store.get(dataset_name)
    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.shape[1] < 2:
        return "Không đủ cột số để tính tương quan."
    corr = numeric_df.corr(method=method)
    return corr.to_string()
 
 
@tool
def hypothesis_test(tool_input: str) -> str:
    """Chạy kiểm định thống kê. Action Input dạng:
    dataset_name=ten, test_type=normality|ttest|anova|chi2, column=ten_cot, group_column=ten_cot_nhom (cần cho ttest/anova/chi2)."""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", "")
    test_type = args.get("test_type", "")
    column = args.get("column", "")
    group_column = args.get("group_column")
    df = store.get(dataset_name)
    try:
        if test_type == "normality":
            stat, p = scipy_stats.shapiro(df[column].dropna())
            return f"Shapiro-Wilk: stat={stat:.4f}, p={p:.4f} -> {'Có vẻ chuẩn' if p > 0.05 else 'Không chuẩn'} (alpha=0.05)"
        elif test_type == "ttest":
            groups = df[group_column].unique()
            if len(groups) != 2:
                return f"T-test cần đúng 2 nhóm, '{group_column}' có {len(groups)} nhóm."
            g1 = df[df[group_column] == groups[0]][column].dropna()
            g2 = df[df[group_column] == groups[1]][column].dropna()
            stat, p = scipy_stats.ttest_ind(g1, g2)
            return f"T-test: stat={stat:.4f}, p={p:.4f} -> {'Khác biệt có ý nghĩa' if p < 0.05 else 'Không khác biệt có ý nghĩa'}"
        elif test_type == "anova":
            groups = [g[column].dropna() for _, g in df.groupby(group_column)]
            stat, p = scipy_stats.f_oneway(*groups)
            return f"ANOVA: stat={stat:.4f}, p={p:.4f} -> {'Khác biệt có ý nghĩa' if p < 0.05 else 'Không khác biệt có ý nghĩa'}"
        elif test_type == "chi2":
            contingency = pd.crosstab(df[column], df[group_column])
            stat, p, dof, _ = scipy_stats.chi2_contingency(contingency)
            return f"Chi-square: stat={stat:.4f}, p={p:.4f}, dof={dof} -> {'Có liên hệ' if p < 0.05 else 'Không có liên hệ'}"
        else:
            return f"test_type '{test_type}' không hợp lệ."
    except Exception as e:
        return f"Lỗi khi test: {str(e)}"
 
 
@tool
def outlier_detection(tool_input: str) -> str:
    """Tìm outlier trong 1 cột số. Action Input dạng:
    dataset_name=ten, column=ten_cot, method=iqr (hoặc zscore, mặc định iqr)."""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", "")
    column = args.get("column", "")
    method = args.get("method", "iqr")
    df = store.get(dataset_name)
    col = df[column].dropna()
    if method == "iqr":
        q1, q3 = col.quantile(0.25), col.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outliers = col[(col < lower) | (col > upper)]
    else:
        z = np.abs(scipy_stats.zscore(col))
        outliers = col[z > 3]
    return f"Tìm thấy {len(outliers)} outlier trong '{column}' (method={method}): {outliers.tolist()[:20]}"
 
 
CHART_OUTPUT_PATH = "/kaggle/working/last_chart.png"

@tool
def create_visualization(tool_input: str) -> str:
    """Tạo biểu đồ. Action Input dạng: dataset_name=ten, chart_type=histogram|scatter|bar|line|box|heatmap|pie,
    x=ten_cot, y=ten_cot, hue=ten_cot (tùy chọn, để phân nhóm màu)."""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", "")
    chart_type = args.get("chart_type", "")
    x = args.get("x") or args.get("column")
    y = args.get("y")
    hue = args.get("hue")
    df = store.get(dataset_name)
    fig, ax = plt.subplots(figsize=(8, 5))
    try:
        if chart_type == "histogram":
            sns.histplot(df[x], ax=ax, kde=True)
        elif chart_type == "scatter":
            sns.scatterplot(data=df, x=x, y=y, hue=hue, ax=ax)
        elif chart_type == "bar":
            df.groupby(x)[y].mean().plot(kind="bar", ax=ax)
        elif chart_type == "line":
            df.plot(x=x, y=y, ax=ax)
        elif chart_type == "box":
            sns.boxplot(data=df, x=x, y=y, ax=ax)
        elif chart_type == "heatmap":
            sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, ax=ax)
        elif chart_type == "pie":
            df[x].value_counts().plot(kind="pie", ax=ax, autopct="%1.1f%%")
        else:
            return f"chart_type '{chart_type}' không hợp lệ."
        ax.set_title(f"{chart_type} - {dataset_name}")
        fig.tight_layout()
        fig.savefig(CHART_OUTPUT_PATH, dpi=120, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        return f"Đã tạo biểu đồ {chart_type}, lưu tại {CHART_OUTPUT_PATH}"
    except Exception as e:
        plt.close(fig)
        return f"Lỗi tạo biểu đồ: {str(e)}"
 
 
@tool
def create_distribution_report(tool_input: str) -> str:
    """Tạo report phân tích phân bố 4-plot (histogram, box, QQ-plot, violin) cho 1 cột.
    Action Input dạng: dataset_name=ten, column=ten_cot"""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", "")
    column = args.get("column", "")
    df = store.get(dataset_name)
    col = df[column].dropna()
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    sns.histplot(col, kde=True, ax=axes[0, 0]); axes[0, 0].set_title("Histogram")
    sns.boxplot(x=col, ax=axes[0, 1]); axes[0, 1].set_title("Boxplot")
    scipy_stats.probplot(col, plot=axes[1, 0]); axes[1, 0].set_title("Q-Q Plot")
    sns.violinplot(x=col, ax=axes[1, 1]); axes[1, 1].set_title("Violin")
    fig.tight_layout()
    fig.savefig(CHART_OUTPUT_PATH, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return f"Đã tạo distribution report cho '{column}'. Mean={col.mean():.2f}, Std={col.std():.2f}, Skew={col.skew():.2f}"
 
 
@tool
def filter_data(tool_input: str) -> str:
    """Filter dữ liệu theo điều kiện pandas query. Action Input dạng:
    dataset_name=ten, condition=age > 30, new_dataset_name=ten_moi"""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", "")
    condition = args.get("condition", "")
    new_dataset_name = args.get("new_dataset_name", "")
    df = store.get(dataset_name)
    try:
        filtered = df.query(condition)
        store.add(new_dataset_name, filtered)
        return f"Đã filter '{dataset_name}' -> '{new_dataset_name}': {len(filtered)}/{len(df)} dòng còn lại"
    except Exception as e:
        return f"Lỗi filter: {str(e)}"
 
 
@tool
def aggregate_data(tool_input: str) -> str:
    """Group & aggregate. Action Input dạng: dataset_name=ten, group_by=ten_cot,
    aggregations=col:func,col2:func2 (vd revenue:sum,profit:mean), new_dataset_name=ten_moi"""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", "")
    group_by = args.get("group_by", "")
    aggregations = args.get("aggregations", "")
    new_dataset_name = args.get("new_dataset_name", "")
    df = store.get(dataset_name)
    try:
        agg_dict = {}
        for pair in aggregations.split(","):
            col, func = pair.split(":")
            agg_dict[col.strip()] = func.strip()
        result = df.groupby(group_by).agg(agg_dict).reset_index()
        store.add(new_dataset_name, result)
        return f"Đã aggregate -> '{new_dataset_name}':\n{result.to_string()}"
    except Exception as e:
        return f"Lỗi aggregate: {str(e)}"
 
 
@tool
def add_calculated_column(tool_input: str) -> str:
    """Thêm cột mới tính từ expression pandas. Action Input dạng:
    dataset_name=ten, new_column=ten_cot_moi, expression=revenue * 0.1"""
    args = _parse_kv_string(tool_input)
    dataset_name = args.get("dataset_name", "")
    new_column = args.get("new_column", "")
    expression = args.get("expression", "")
    df = store.get(dataset_name)
    try:
        df[new_column] = df.eval(expression)
        store.add(dataset_name, df)
        return f"Đã thêm cột '{new_column}' vào '{dataset_name}'"
    except Exception as e:
        return f"Lỗi tính cột: {str(e)}"
 
 
@tool
def generate_summary_report(dataset_name: str) -> str:
    """Tạo báo cáo tổng hợp đầy đủ cho 1 dataset: shape, dtypes, missing values, describe."""
    df = store.get(dataset_name)
    report = [
        f"=== REPORT: {dataset_name} ===",
        f"Kích thước: {df.shape[0]} dòng x {df.shape[1]} cột",
        f"Dtypes:\n{df.dtypes.to_string()}",
        f"Missing values:\n{df.isnull().sum().to_string()}",
        f"Describe:\n{df.describe().to_string()}",
    ]
    return "\n\n".join(report)
 
 
@tool
def get_analysis_history() -> str:
    """Xem lịch sử các hành động đã thực hiện trong session."""
    if not store.history:
        return "Chưa có lịch sử nào."
    return "\n".join(f"{i+1}. {h}" for i, h in enumerate(store.history))
 
print("✅ Tools đã định nghĩa xong")

In [ ]:
from langchain_core.prompts import PromptTemplate
 
REACT_TEMPLATE = """Trả lời câu hỏi sau bằng cách dùng các tool có sẵn. Bạn có quyền dùng các tool sau:
 
{tools}
 
Dùng đúng format sau:
 
Question: câu hỏi cần trả lời
Thought: suy nghĩ về việc cần làm gì tiếp theo
Action: tên 1 tool trong [{tool_names}]
Action Input: tham số cho tool đó (dạng key=value, cách nhau bởi dấu phẩy)
Observation: kết quả của tool
... (lặp lại Thought/Action/Action Input/Observation nếu cần)
Thought: tôi đã có đủ thông tin để trả lời
Final Answer: câu trả lời cuối cùng cho người dùng
 
Bắt đầu!
 
Question: {input}
Thought:{agent_scratchpad}"""
 
react_prompt = PromptTemplate.from_template(REACT_TEMPLATE)
 
# QUAN TRỌNG: llm.bind(stop=[...]) KHÔNG có tác dụng thật với HuggingFacePipeline
# trong bản langchain_huggingface đang dùng (_call() không implement enforce_stop_tokens
# cho stop kwarg) -> model generate tới đâu cũng được, tự bịa luôn Observation + Final
# Answer trong 1 lần. Phải tự cắt chuỗi bằng Python NGAY SAU khi llm trả về, độc lập
# hoàn toàn với cơ chế stop của LangChain.
from langchain_core.runnables import RunnableLambda
 
STOP_TOKENS = ["\nObservation:", "\nObservation"]
 
def _truncate_at_stop(text: str) -> str:
    """Cắt bỏ mọi thứ từ stop token đầu tiên trở đi (model hay tự bịa Observation/Final
    Answer giả sau đó). Lấy điểm cắt SỚM NHẤT trong số các stop token tìm được."""
    cut = len(text)
    for s in STOP_TOKENS:
        idx = text.find(s)
        if idx != -1:
            cut = min(cut, idx)
    return text[:cut]
 
llm_react = llm | RunnableLambda(_truncate_at_stop)

In [ ]:
from langchain_classic.agents import create_react_agent, AgentExecutor
 
def build_specialist(tools, max_iterations=4):
    agent = create_react_agent(llm_react, tools, react_prompt)
    return AgentExecutor(
        agent=agent, tools=tools,
        verbose=True, max_iterations=max_iterations,
        # Thay handle_parsing_errors=True (mặc định sẽ nhồi NGUYÊN VĂN lỗi + raw output
        # hallucinate của model ngược vào agent_scratchpad -> model đời sau dễ "copy lại"
        # y chang đoạn rác đó -> trông như loop vô hạn) bằng 1 message ngắn, rõ ràng,
        # buộc model tự sửa format thay vì lặp lại rác cũ.
        handle_parsing_errors=(
            "Lỗi format: bạn chỉ được viết MỘT Action (kèm Action Input) "
            "HOẶC MỘT Final Answer trong 1 lượt, không được viết cả hai. "
            "Hãy thử lại đúng format, ngắn gọn."
        ),
    )
 
data_loader_executor = build_specialist([load_csv, create_sample_dataset, list_available_datasets])
statistician_executor = build_specialist([describe_dataset, correlation_analysis, hypothesis_test, outlier_detection])
visualizer_executor = build_specialist([create_visualization, create_distribution_report])
transformer_executor = build_specialist([filter_data, aggregate_data, add_calculated_column])
reporter_executor = build_specialist([generate_summary_report, get_analysis_history])
 
print("✅ 5 specialist agents đã sẵn sàng")

In [ ]:
@tool
def data_loader_agent(task: str) -> str:
    """Giao việc liên quan tới load/tạo dataset cho chuyên gia data_loader. task: mô tả việc cần làm."""
    return data_loader_executor.invoke({"input": task})["output"]
 
@tool
def statistician_agent(task: str) -> str:
    """Giao việc phân tích thống kê (describe, correlation, hypothesis test, outlier) cho chuyên gia statistician."""
    return statistician_executor.invoke({"input": task})["output"]
 
@tool
def visualizer_agent(task: str) -> str:
    """Giao việc vẽ biểu đồ cho chuyên gia visualizer.
    task: mô tả bằng ngôn ngữ tự nhiên, ví dụ:
    'Vẽ histogram cột weight_kg của dataset df_fifa'
    'Vẽ scatter giữa height_cm và weight_kg của df_fifa'
    KHÔNG dùng key=value trong task. Chỉ mô tả tự nhiên."""
    return visualizer_executor.invoke({"input": task})["output"]
 
@tool
def transformer_agent(task: str) -> str:
    """Giao việc biến đổi dữ liệu (filter, aggregate, tính cột mới) cho chuyên gia transformer."""
    return transformer_executor.invoke({"input": task})["output"]
 
@tool
def reporter_agent(task: str) -> str:
    """Giao việc tạo báo cáo tổng hợp cho chuyên gia reporter."""
    return reporter_executor.invoke({"input": task})["output"]
 
orchestrator_tools = [data_loader_agent, statistician_agent, visualizer_agent, transformer_agent, reporter_agent]
master_agent = create_react_agent(llm_react, orchestrator_tools, react_prompt)
master_executor = AgentExecutor(
    agent=master_agent, tools=orchestrator_tools,
    verbose=True, max_iterations=8,
    handle_parsing_errors=(
        "Lỗi format: bạn chỉ được viết MỘT Action (kèm Action Input) "
        "HOẶC MỘT Final Answer trong 1 lượt, không được viết cả hai. "
        "Hãy thử lại đúng format, ngắn gọn."
    ),
)
 
print("✅ Master orchestrator sẵn sàng - gọi analyze('câu hỏi') để dùng")

In [ ]:
def analyze(query: str) -> str:
    # master_executor.invoke() mỗi lần là 1 conversation MỚI, không có chat_history.
    # Nếu không nhắc lại, orchestrator ở lần gọi sau sẽ không biết dataset nào đã
    # load từ lần gọi trước (tên gì) -> dễ tự bịa tên hoặc giao lại việc load từ đầu
    # nhưng không có path thật -> tool không chạy được -> store rỗng.
    # Nên chủ động nhắc danh sách dataset hiện có vào đầu mỗi query.
    if store.datasets:
        existing = ", ".join(f"'{name}'" for name in store.datasets.keys())
        context = f"(Các dataset đã có sẵn trong hệ thống: {existing}. Dùng lại tên này, KHÔNG cần load lại.) "
    else:
        context = "(Chưa có dataset nào trong hệ thống.) "
 
    try:
        result = master_executor.invoke({"input": context + query})
        return result["output"]
    except Exception as e:
        return f"Lỗi: {str(e)}"
 
 
def debug_store() -> None:
    """Gọi hàm này bất cứ lúc nào để kiểm tra thực tế dataset nào ĐANG có trong store
    (độc lập với những gì agent 'nói' nó đã làm) - hữu ích để phát hiện hallucination."""
    if not store.datasets:
        print("⚠️ store.datasets RỖNG - chưa có dataset nào thực sự được load/tạo.")
    else:
        for name, df in store.datasets.items():
            print(f"- '{name}': {df.shape[0]} dòng x {df.shape[1]} cột -> {list(df.columns)}")

In [ ]:
import traceback
try:
    result = master_executor.invoke({"input": "Tạo dataset sales mẫu, tên là sales_data"})
    print(result)
except Exception:
    traceback.print_exc()

In [ ]:
debug_store()

In [ ]:
print(analyze("Describe 'sales_data' và vẽ histogram của revenue"))

In [ ]:
debug_store()

In [ ]:
# Cell riêng: load thẳng, không qua agent
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/rauffauzanrambe/fifa-world-cup-2026-player-performance-dataset/fifa_world_cup_2026_player_performance.csv")
store.add("df_fifa", df)
print(f"✅ Loaded fifa: {df.shape}, columns: {list(df.columns)}")

In [ ]:
print(analyze("""
Vẽ histogram cân nặng (cột 'weight_kg') và chiều cao (cột 'height_cm') 
của các cầu thủ trong dataset 'df_fifa'. 
Gọi visualizer_agent 2 lần riêng biệt với JSON input:
Lần 1: {"dataset_name": "df_fifa", "chart_type": "histogram", "x": "weight_kg"}
Lần 2: {"dataset_name": "df_fifa", "chart_type": "histogram", "x": "height_cm"}
"""))